# 07 — Full Architecture + Rich Pooling + Random Val

**Amaç:** 06_redesign'da sade GNN (homojen, no multi-task, no pretrain) + rich pool + random val ile test AUC=0.9140 elde ettik (MLP 0.9326'ya yakın).

Bu notebook **04_model'in tam mimarisini** (4 edge type RGCN + multi-task + pretrain) rich pool + random val protokolüyle test eder. Soru: ablation'da kaybeden bileşenler, rich pool + random val ile birlikte değerli mi?

## Karşılaştırma

| Konfigürasyon | Edges | Pretrain | Multi-task | Pooling | Val protokol | Test AUC |
|---|---|---|---|---|---|---|
| 04_model (orijinal) | 4 (RGCN) | ✓ | ✓ | mean+bilinear | temporal | 0.8165 |
| 06 sade + rich+rand | 1 (GCN) | ✗ | ✗ | rich+bilinear | random | 0.9140 |
| **07 (bu notebook)** | 4 (RGCN) | ✓ | ✓ | rich+bilinear | random | **?** |
| MLP baseline | — | — | — | rich raw | random | 0.9326 |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/coalition_gnn'
SNAP_DIR = os.path.join(PROJECT_DIR, 'data/snapshots')
COAL_DIR = os.path.join(PROJECT_DIR, 'data/coalitions')
MODELS_DIR = os.path.join(PROJECT_DIR, 'models')

START_YEAR, END_YEAR = 1946, 2016
TRAIN_END = 1999
VAL_END = 2009

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from tqdm.auto import tqdm
import random

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

EDGE_TYPES = ['allies', 'trades', 'votes_with', 'conflicts_with']
NUM_RELATIONS = len(EDGE_TYPES)

In [ ]:
# Snapshot'ları yükle (heterojen, 4 edge type — 04_model ile aynı)
def load_snapshot(year):
    path = os.path.join(SNAP_DIR, f'graph_{year}.pt')
    if not os.path.exists(path):
        return None
    data = torch.load(path, weights_only=False)
    x = data['country'].x
    cow_codes = data['country'].cow_code.numpy()
    
    edge_index_list, edge_type_list = [], []
    for rel_idx, rel_name in enumerate(EDGE_TYPES):
        key = ('country', rel_name, 'country')
        if key in data.edge_types:
            ei = data[key].edge_index
            edge_index_list.append(ei)
            edge_type_list.append(torch.full((ei.shape[1],), rel_idx, dtype=torch.long))
    
    if not edge_index_list:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_type = torch.zeros(0, dtype=torch.long)
    else:
        edge_index = torch.cat(edge_index_list, dim=1)
        edge_type = torch.cat(edge_type_list)
    
    return {
        'x': x.to(DEVICE),
        'edge_index': edge_index.to(DEVICE),
        'edge_type': edge_type.to(DEVICE),
        'cow_codes': cow_codes,
        'code_to_idx': {int(c): i for i, c in enumerate(cow_codes)},
        'year': year,
    }

snapshots = {}
for year in range(START_YEAR, END_YEAR + 1):
    snap = load_snapshot(year)
    if snap is not None:
        snapshots[year] = snap

NUM_FEATURES = next(iter(snapshots.values()))['x'].shape[1]
print(f'Snapshot: {len(snapshots)} yıl, {NUM_FEATURES} özellik, {NUM_RELATIONS} ilişki tipi')

In [ ]:
# Koalisyon örnekleri
pos_df = pd.read_parquet(os.path.join(COAL_DIR, 'positive_samples.parquet'))
neg_df = pd.read_parquet(os.path.join(COAL_DIR, 'negative_samples.parquet'))

def parse_members(s):
    if isinstance(s, list): return [int(x) for x in s]
    if isinstance(s, str): return [int(x) for x in s.split(',')]
    return []

def build_samples(df, label):
    samples = []
    for _, row in df.iterrows():
        year = int(row['year'])
        if year not in snapshots: continue
        members = parse_members(row['members_str'])
        c2i = snapshots[year]['code_to_idx']
        valid = [m for m in members if m in c2i]
        if len(valid) < 2: continue
        samples.append({
            'year': year,
            'members': valid,
            'member_idx': [c2i[m] for m in valid],
            'label': label,
            'duration': float(row.get('total_duration', 0)) if label == 1 else 0.0,
            'cohesion': float(row.get('internal_vote_agreement', 0) or 0) if label == 1 else 0.0,
        })
    return samples

all_samples = build_samples(pos_df, 1) + build_samples(neg_df, 0)

# Test (2010-2016) ayrı — temporal
test_samples = [s for s in all_samples if s['year'] > VAL_END]
# Train+Val birleştir (1946-2009), içinden random %15 val
pre_test = [s for s in all_samples if s['year'] <= VAL_END]

rng = np.random.default_rng(SEED)
idx = np.arange(len(pre_test))
rng.shuffle(idx)
val_size = int(0.15 * len(pre_test))
val_set = set(idx[:val_size].tolist())

train_samples = [pre_test[i] for i in range(len(pre_test)) if i not in val_set]
val_samples = [pre_test[i] for i in range(len(pre_test)) if i in val_set]

print(f'Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}')
print(f'Train pos: {np.mean([s["label"] for s in train_samples]):.3f}')
print(f'Val pos:   {np.mean([s["label"] for s in val_samples]):.3f}')
print(f'Test pos:  {np.mean([s["label"] for s in test_samples]):.3f}')

## Full Mimari: RGCN encoder + Rich Pool + Multi-task heads

In [ ]:
class RGCNEncoder(nn.Module):
    """04_model ile aynı encoder."""
    def __init__(self, in_channels, hidden_channels, out_channels, num_relations, dropout=0.2):
        super().__init__()
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations)
        self.conv2 = RGCNConv(hidden_channels, out_channels, num_relations)
        self.dropout = dropout
        self.norm1 = nn.LayerNorm(hidden_channels)
        self.norm2 = nn.LayerNorm(out_channels)
    
    def forward(self, x, edge_index, edge_type):
        h = self.conv1(x, edge_index, edge_type)
        h = self.norm1(h); h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.conv2(h, edge_index, edge_type)
        h = self.norm2(h)
        return h


class RichCoalitionHead(nn.Module):
    """Rich pooling (mean+std+max+min+size) + bilinear + multi-task heads."""
    def __init__(self, embed_dim, raw_dim, hidden_dim=128, dropout=0.3):
        super().__init__()
        # Pool boyutu: 4*(embed+raw) + 1 size + 1 pair
        pool_dim = 4 * (embed_dim + raw_dim) + 2
        
        self.W_pair = nn.Parameter(torch.randn(embed_dim, embed_dim) * 0.01)
        
        # Validity head
        self.validity_mlp = nn.Sequential(
            nn.Linear(pool_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )
        
        # Multi-task heads (sadece embedding pool üzerinden, raw kullanmaz)
        embed_pool_dim = 4 * embed_dim + 1
        self.duration_head = nn.Sequential(
            nn.Linear(embed_pool_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1), nn.Softplus(),
        )
        self.cohesion_head = nn.Sequential(
            nn.Linear(embed_pool_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1), nn.Sigmoid(),
        )
    
    def rich_pool(self, feats):
        if feats.shape[0] > 1:
            std = feats.std(dim=0)
        else:
            std = torch.zeros_like(feats[0])
        return torch.cat([feats.mean(dim=0), std, feats.max(dim=0).values, feats.min(dim=0).values])
    
    def forward(self, z_members, x_raw_members):
        k = z_members.shape[0]
        size_feat = torch.tensor([k / 20.0], device=z_members.device)
        
        z_pool = self.rich_pool(z_members)
        raw_pool = self.rich_pool(x_raw_members)
        
        if k >= 2:
            Wz = z_members @ self.W_pair
            pm = z_members @ Wz.T
            off_diag = pm.sum() - pm.diag().sum()
            pair_score = off_diag / (k * (k - 1))
        else:
            pair_score = torch.tensor(0.0, device=z_members.device)
        
        validity_input = torch.cat([z_pool, raw_pool, size_feat, pair_score.unsqueeze(0)])
        validity = self.validity_mlp(validity_input).squeeze()
        
        # Multi-task: sadece embedding pool + size
        mt_input = torch.cat([z_pool, size_feat])
        duration = self.duration_head(mt_input).squeeze()
        cohesion = self.cohesion_head(mt_input).squeeze()
        
        return validity, duration, cohesion


class FullRichGNN(nn.Module):
    def __init__(self, num_features, hidden_dim=128, embed_dim=128,
                 num_relations=4, head_hidden=128, dropout=0.3):
        super().__init__()
        self.encoder = RGCNEncoder(num_features, hidden_dim, embed_dim, num_relations, dropout)
        self.head = RichCoalitionHead(embed_dim, num_features, head_hidden, dropout)
    
    def forward_sample(self, snap, member_idx_list):
        z_all = self.encoder(snap['x'], snap['edge_index'], snap['edge_type'])
        idx = torch.tensor(member_idx_list, dtype=torch.long, device=DEVICE)
        z = z_all[idx]
        raw = snap['x'][idx]
        return self.head(z, raw)


model = FullRichGNN(NUM_FEATURES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Full+Rich GNN parametreleri: {n_params:,}')

In [ ]:
# Pretrained encoder yükle
pretrain_path = os.path.join(MODELS_DIR, 'encoder_pretrained.pt')
if os.path.exists(pretrain_path):
    ckpt = torch.load(pretrain_path, weights_only=False)
    if 'encoder_state_dict' in ckpt:
        model.encoder.load_state_dict(ckpt['encoder_state_dict'])
    else:
        model.encoder.load_state_dict(ckpt)
    print('✓ Pretrained encoder yüklendi')
else:
    print('⚠ Pretrain dosyası bulunamadı — random init ile devam')

In [ ]:
# Eğitim hiperparametreleri
EPOCHS = 150
LR = 1e-3
ENCODER_LR = 1e-4
WEIGHT_DECAY = 1e-3
LABEL_SMOOTHING = 0.1
PATIENCE = 25
POS_WEIGHT = 2.5
BATCH_SIZE = 32
LAMBDA_DUR = 0.3
LAMBDA_COH = 0.3

# Encoder ve head farklı LR (04_model staged fine-tuning ile aynı mantık)
optimizer = torch.optim.Adam([
    {'params': model.encoder.parameters(), 'lr': ENCODER_LR},
    {'params': model.head.parameters(), 'lr': LR},
], weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def bce_smooth(logit, target, smoothing=0.1):
    target_s = target * (1 - smoothing) + 0.5 * smoothing
    w = torch.tensor(POS_WEIGHT if target.item() > 0.5 else 1.0, device=DEVICE)
    return F.binary_cross_entropy_with_logits(logit, target_s, weight=w)

def evaluate(model, samples):
    model.eval()
    logits, labels = [], []
    with torch.no_grad():
        for s in samples:
            v, _, _ = model.forward_sample(snapshots[s['year']], s['member_idx'])
            logits.append(v.item()); labels.append(s['label'])
    logits, labels = np.array(logits), np.array(labels)
    probs = 1/(1+np.exp(-logits))
    p, r, _ = precision_recall_curve(labels, probs)
    f1 = (2*p*r/(p+r+1e-10)).max()
    return {'AUC': roc_auc_score(labels, probs),
            'PR-AUC': average_precision_score(labels, probs),
            'F1_opt': f1}

In [ ]:
best_val_auc, best_state, pat = 0, None, 0
history = []

for epoch in range(EPOCHS):
    model.train()
    random.shuffle(train_samples)
    epoch_loss, nb = 0.0, 0
    
    for i in range(0, len(train_samples), BATCH_SIZE):
        batch = train_samples[i:i+BATCH_SIZE]
        optimizer.zero_grad()
        
        bl, nv = 0.0, 0
        for s in batch:
            validity, dur, coh = model.forward_sample(snapshots[s['year']], s['member_idx'])
            target = torch.tensor(float(s['label']), device=DEVICE)
            loss = bce_smooth(validity, target, LABEL_SMOOTHING)
            
            # Multi-task: sadece pozitif örnekler için
            if s['label'] == 1:
                dur_target = torch.log1p(torch.tensor(s['duration'], device=DEVICE))
                loss = loss + LAMBDA_DUR * F.mse_loss(dur, dur_target)
                if s['cohesion'] > 0:
                    coh_target = torch.tensor(s['cohesion'], device=DEVICE)
                    loss = loss + LAMBDA_COH * F.mse_loss(coh, coh_target)
            
            bl = bl + loss; nv += 1
        
        if nv == 0: continue
        bl = bl / nv
        bl.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += bl.item(); nb += 1
    
    scheduler.step()
    train_loss = epoch_loss / max(nb, 1)
    vm = evaluate(model, val_samples)
    history.append({'epoch': epoch, 'train_loss': train_loss, **vm})
    
    if vm['AUC'] > best_val_auc:
        best_val_auc = vm['AUC']
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        pat = 0; flag = ' ←'
    else:
        pat += 1; flag = ''
    
    if epoch % 5 == 0 or flag:
        print(f"Epoch {epoch:3d} | loss={train_loss:.4f} | val AUC={vm['AUC']:.4f} F1={vm['F1_opt']:.4f}{flag}")
    
    if pat >= PATIENCE:
        print(f'\nEarly stop ep{epoch}')
        break

model.load_state_dict(best_state)
torch.save(best_state, os.path.join(MODELS_DIR, 'full_rich_gnn.pt'))
print(f'\n✓ En iyi val AUC: {best_val_auc:.4f}')

In [ ]:
test_metrics = evaluate(model, test_samples)
val_metrics = evaluate(model, val_samples)

print('='*70)
print('FULL ARCHITECTURE + RICH POOL + RANDOM VAL — SONUÇLAR')
print('='*70)
print(f"  Val  (random):  AUC={val_metrics['AUC']:.4f}  PR-AUC={val_metrics['PR-AUC']:.4f}  F1={val_metrics['F1_opt']:.4f}")
print(f"  Test (2010-16): AUC={test_metrics['AUC']:.4f}  PR-AUC={test_metrics['PR-AUC']:.4f}  F1={test_metrics['F1_opt']:.4f}")
print('='*70)
print('\nTÜM KARŞILAŞTIRMA:')
print(f"  Full GNN (orijinal, temporal):     AUC=0.8165  F1=0.6283")
print(f"  Sade GNN (06, temporal val):       AUC=0.8234  F1=0.6818")
print(f"  Sade GNN + rich pool + random val: AUC=0.9140  F1=0.7792")
print(f"  MLP baseline (random val):         AUC=0.9326  F1=0.7911")
print(f"  ★ Full GNN + rich pool + random:   AUC={test_metrics['AUC']:.4f}  F1={test_metrics['F1_opt']:.4f}")

## Yorumlama

| Sonuç | Anlam |
|-------|-------|
| Full+rich > sade+rich (>0.92) | Ablation'da kaybeden bileşenler aslında rich pool ile birlikte değerli — orijinal mimariye geri dön |
| Full+rich ≈ sade+rich (~0.91) | Ekstra bileşenler nötr — basitlik için sade modeli tut |
| Full+rich < sade+rich (<0.90) | Ablation bulguları doğru — sade model + rich pool en iyisi |

## Bootstrap Confidence Intervals

GNN (Full+Rich) vs MLP baseline farkının istatistiksel anlamlılığını ölçer.

**Yöntem:**
1. Test setinden 1000 kez yerine-koyarak (with replacement) resample
2. Her resample'da her iki modelin AUC/F1/PR-AUC'sini hesapla
3. **Paired bootstrap:** Her resample'da Δ = GNN_metric - MLP_metric kaydet
4. %2.5 ve %97.5 yüzdelikleri → %95 güven aralığı

**Yorumlama:**
- Δ aralığı tamamen pozitifse → GNN istatistiksel olarak üstün
- Δ aralığı 0'ı içeriyorsa → fark anlamlı değil

In [ ]:
# 1. GNN test tahminleri (zaten eğitilmiş 'model' kullanılır)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

model.eval()
gnn_test_logits = []
test_labels_list = []

with torch.no_grad():
    for s in test_samples:
        v, _, _ = model.forward_sample(snapshots[s['year']], s['member_idx'])
        gnn_test_logits.append(v.item())
        test_labels_list.append(s['label'])

gnn_test_logits = np.array(gnn_test_logits)
test_labels_arr = np.array(test_labels_list)
gnn_test_probs = 1 / (1 + np.exp(-gnn_test_logits))

print(f'GNN test predictions: {len(gnn_test_probs)} örnek')
print(f'  AUC={roc_auc_score(test_labels_arr, gnn_test_probs):.4f}')

# 2. MLP baseline'i 05_ablation ile aynı feature setiyle yeniden eğit
def extract_rich_features(sample):
    snap = snapshots[sample['year']]
    x = snap['x'].cpu().numpy()
    members = sample['member_idx']
    mf = x[members]  # (k, D)
    return np.concatenate([
        mf.mean(axis=0),
        mf.std(axis=0) if len(members) > 1 else np.zeros(mf.shape[1]),
        mf.max(axis=0),
        mf.min(axis=0),
        [len(members) / 20.0]
    ])

# Train+Val (1946-2009) → MLP training set
all_pre_test = train_samples + val_samples
X_combined = np.array([extract_rich_features(s) for s in all_pre_test])
y_combined = np.array([s['label'] for s in all_pre_test])
X_test_baseline = np.array([extract_rich_features(s) for s in test_samples])

# NaN temizle + standardize
X_combined = np.nan_to_num(X_combined, 0)
X_test_baseline = np.nan_to_num(X_test_baseline, 0)
scaler = StandardScaler()
X_combined = scaler.fit_transform(X_combined)
X_test_baseline = scaler.transform(X_test_baseline)

# MLP eğit (05_ablation ile aynı ayarlar)
mlp = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500,
                    early_stopping=True, validation_fraction=0.15,
                    random_state=SEED, alpha=1e-4)
mlp.fit(X_combined, y_combined)
mlp_test_probs = mlp.predict_proba(X_test_baseline)[:, 1]

print(f'\nMLP test predictions: {len(mlp_test_probs)} örnek')
print(f'  AUC={roc_auc_score(test_labels_arr, mlp_test_probs):.4f}')


In [ ]:
# 3. Bootstrap CI hesabı
N_BOOT = 1000
n_test = len(test_labels_arr)

rng_boot = np.random.default_rng(SEED)

def metrics_for_subset(probs, labels):
    """AUC, PR-AUC, F1 hesapla. Tek sınıf varsa NaN döndür."""
    if len(np.unique(labels)) < 2:
        return np.nan, np.nan, np.nan
    auc = roc_auc_score(labels, probs)
    pr_auc = average_precision_score(labels, probs)
    p, r, _ = precision_recall_curve(labels, probs)
    f1 = (2*p*r/(p+r+1e-10)).max()
    return auc, pr_auc, f1

gnn_aucs, mlp_aucs = [], []
gnn_prs, mlp_prs = [], []
gnn_f1s, mlp_f1s = [], []
delta_aucs, delta_prs, delta_f1s = [], [], []

for b in range(N_BOOT):
    idx_boot = rng_boot.integers(0, n_test, size=n_test)
    y_b = test_labels_arr[idx_boot]
    gnn_p_b = gnn_test_probs[idx_boot]
    mlp_p_b = mlp_test_probs[idx_boot]

    gnn_auc, gnn_pr, gnn_f1 = metrics_for_subset(gnn_p_b, y_b)
    mlp_auc, mlp_pr, mlp_f1 = metrics_for_subset(mlp_p_b, y_b)

    if not np.isnan(gnn_auc) and not np.isnan(mlp_auc):
        gnn_aucs.append(gnn_auc); mlp_aucs.append(mlp_auc)
        gnn_prs.append(gnn_pr);   mlp_prs.append(mlp_pr)
        gnn_f1s.append(gnn_f1);   mlp_f1s.append(mlp_f1)
        # Paired (aynı resample'da fark)
        delta_aucs.append(gnn_auc - mlp_auc)
        delta_prs.append(gnn_pr - mlp_pr)
        delta_f1s.append(gnn_f1 - mlp_f1)

def ci(arr, level=95):
    lo = np.percentile(arr, (100 - level) / 2)
    hi = np.percentile(arr, 100 - (100 - level) / 2)
    return np.mean(arr), lo, hi

print('='*70)
print(f'BOOTSTRAP %95 CI (N={N_BOOT} resample, n={n_test} test örnek)')
print('='*70)

for name, gnn, mlp, delta in [
    ('AUC',    gnn_aucs, mlp_aucs, delta_aucs),
    ('PR-AUC', gnn_prs,  mlp_prs,  delta_prs),
    ('F1_opt', gnn_f1s,  mlp_f1s,  delta_f1s),
]:
    g_mean, g_lo, g_hi = ci(gnn)
    m_mean, m_lo, m_hi = ci(mlp)
    d_mean, d_lo, d_hi = ci(delta)

    # Δ aralığı 0'ı içeriyor mu?
    significant = (d_lo > 0) or (d_hi < 0)
    sig_str = ' ★ ANLAMLI' if significant else '   (0 dahil)'

    print(f'\n  {name}:')
    print(f'    GNN:  {g_mean:.4f}  [{g_lo:.4f}, {g_hi:.4f}]')
    print(f'    MLP:  {m_mean:.4f}  [{m_lo:.4f}, {m_hi:.4f}]')
    print(f'    Δ:    {d_mean:+.4f}  [{d_lo:+.4f}, {d_hi:+.4f}]{sig_str}')

print('\n' + '='*70)


In [ ]:
# 4. Görselleştirme: bootstrap dağılımları
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, name, gnn, mlp, delta in [
    (axes[0], 'AUC',    gnn_aucs, mlp_aucs, delta_aucs),
    (axes[1], 'PR-AUC', gnn_prs,  mlp_prs,  delta_prs),
    (axes[2], 'F1',     gnn_f1s,  mlp_f1s,  delta_f1s),
]:
    ax.hist(gnn, bins=40, alpha=0.55, color='steelblue', label='Full GNN+Rich')
    ax.hist(mlp, bins=40, alpha=0.55, color='darkorange', label='MLP baseline')

    g_lo, g_hi = np.percentile(gnn, [2.5, 97.5])
    m_lo, m_hi = np.percentile(mlp, [2.5, 97.5])
    ax.axvspan(g_lo, g_hi, alpha=0.10, color='steelblue')
    ax.axvspan(m_lo, m_hi, alpha=0.10, color='darkorange')

    ax.set_title(f'{name} — Bootstrap distribution')
    ax.set_xlabel(name)
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'bootstrap_ci.png'), dpi=120)
plt.show()

# Paired diff histogramı
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name, delta in [
    (axes[0], 'ΔAUC',    delta_aucs),
    (axes[1], 'ΔPR-AUC', delta_prs),
    (axes[2], 'ΔF1',     delta_f1s),
]:
    ax.hist(delta, bins=40, color='seagreen', alpha=0.75)
    ax.axvline(0, color='red', ls='--', alpha=0.7, label='No difference')
    d_lo, d_hi = np.percentile(delta, [2.5, 97.5])
    ax.axvspan(d_lo, d_hi, alpha=0.15, color='seagreen', label='%95 CI')
    ax.set_title(f'Paired {name} (GNN - MLP)')
    ax.set_xlabel(f'{name}')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'bootstrap_diff_ci.png'), dpi=120)
plt.show()

print('Grafikler kaydedildi: bootstrap_ci.png, bootstrap_diff_ci.png')
